In [37]:
import torch
import numpy as np
import random
import math

In [28]:
import torch

# Standard initializations typically range between -1 and 1
x1 = torch.tensor([2.0], requires_grad=True, dtype=torch.float64)
w1 = torch.tensor([-0.5], requires_grad=True, dtype=torch.float64)

x2 = torch.tensor([0.0], requires_grad=True, dtype=torch.float64)
w2 = torch.tensor([1.0], requires_grad=True, dtype=torch.float64)

b = torch.tensor([0.8813735870195432], requires_grad=True, dtype=torch.float64)

# Forward pass: n = (2.0 * -0.5) + (0.0 * 1.0) + 0.8813735... = -0.118626...
n = (x1 * w1 + x2 * w2) + b
o = torch.tanh(n)

print("Output o:", o.data.item())

# Backward pass
o.backward()

print("--- Gradients ---")
print("x2 grad:", x2.grad.item())
print("w2 grad:", w2.grad.item())
print("x1 grad:", x1.grad.item())
print("w1 grad:", w1.grad.item())

Output o: -0.1180730815219694
--- Gradients ---
x2 grad: 0.9860587474199064
w2 grad: 0.0
x1 grad: -0.4930293737099532
w1 grad: 1.9721174948398128


In [76]:
import math

class value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"value(data={self.data})"

    # ---------------- Arithmetic ----------------

    def __add__(self, other):
        other = other if isinstance(other, value) else value(other)

        out = value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __mul__(self, other):
        other = other if isinstance(other, value) else value(other)

        out = value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Only int/float powers supported."

        out = value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = other if isinstance(other, value) else value(other)
        return self * (other ** -1)

    def __rtruediv__(self, other):
        other = other if isinstance(other, value) else value(other)
        return other / self

    # ---------------- Activations ----------------

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)

        out = value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    # ---------------- Backpropagation ----------------

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0

        for node in reversed(topo):
            node._backward()

In [77]:
from graphviz import Digraph
import os
os.environ["PATH"] += os.pathsep + r'C:\Program Files\Graphviz\bin'

def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label = n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [96]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = value(random.uniform(-1, 1))

    def __call__(self, x):
        x = [xi if isinstance(xi, value) else value(xi) for xi in x]

        # W*x + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

In [97]:
x = [2.0, 4.0]
n = Neuron(2)
n(x)

value(data=-0.99889511407841)

In [98]:
# Multilayer neuron 

a = [2.0, 7.0, 4.0]
n = Layer(3, 2)
n(a)

[value(data=0.42853724058319215), value(data=-0.9999992714889115)]

In [99]:
# Multilayer Perceptron 

x = [2.0, 7.0, 4.0]
n = MLP(3, [4, 4, 1])
n(x)

value(data=-0.2168671730814708)

In [100]:
draw_dot(n(x))s

SyntaxError: invalid syntax (4090761515.py, line 1)

In [106]:
xs = [
[2.0, 3.0, -1.0],
[3.0, -1.0, 0.5],
[0.5, 1.0, 1.0],
[1.0, 1.0,-1.0]
]

ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [117]:
for k in range(100):
    # Forward pass 
    ypred = [n(x) for x in xs]
    loss = sum([((yout - ygt)**2) for yout, ygt in zip(ys, ypred)])

    # Backward pass 
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    # Update
    for p in n.parameters():
        p.data += -0.01 * p.grad

    print(k, loss.data)

0 0.055486702204953206
1 0.05482707555496258
2 0.05418178890612706
3 0.053550398391775014
4 0.05293247777915959
5 0.05232761762044036
6 0.05173542445142648
7 0.05115552003502086
8 0.05058754064652392
9 0.05003113639815748
10 0.04948597060035638
11 0.048951719157544045
12 0.04842806999627212
13 0.047914722523744194
14 0.047411387114884046
15 0.04691778462623006
16 0.046433645935056535
17 0.04595871150222604
18 0.04549273095737853
19 0.045035462705154766
20 0.04458667355123488
21 0.04414613834705399
22 0.04371363965212937
23 0.04328896741300119
24 0.04287191865785345
25 0.04246229720593926
26 0.04205991339099113
27 0.04166458379784535
28 0.04127613101156015
29 0.04089438337834876
30 0.04051917477769178
31 0.04015034440503004
32 0.03978773656447758
33 0.039431200471023456
34 0.039080590061728385
35 0.03873576381544572
36 0.0383965845806274
37 0.03806291941079957
38 0.03773463940731626
39 0.037411619569023634
40 0.03709373864848546
41 0.03678087901444351
42 0.0364729265202013
43 0.03616977

In [118]:
ypred

[value(data=0.9403881005533181),
 value(data=-0.896384840540012),
 value(data=-0.9542293267923063),
 value(data=0.9100005602107669)]